In [ ]:
# ==========================================================================
# 농기계 고장·정비 기록 문서 분류 웹 화면 (Gradio) — 셀 하나로 실행
# --------------------------------------------------------------------------
# [전체 흐름]
#   0. 도구 설치          : gradio, koreanize-matplotlib
#   1. 전처리 함수        : 수리내역 + 비고 합치기 → [수리중] 같은 표시·사람 이름 지우기
#                           → 표기 통일(밸트→벨트 등) → 수량·부품 번호·기호 지우기
#   2. 학습 함수 (train)  : 정답 파일과 합치기 → 중복 문서 제거 → 학습 80% / 평가 20% 나누기
#                           → 글자 단위 TF-IDF + 로지스틱 회귀(균형) 학습 → 성적표·혼동 행렬
#   3. 응답 함수 (ask)    : 새 수리 기록 → 같은 정제 → 유형별 확률(응답)
#                           → 과거 기록 중 가장 비슷한 5개 찾기(검색 결과)
#   4. 웹 화면 실행       : ① 파일 선택 → ② 학습 결과 → ③ 기록 분류·검색 순서로 배치
#
# [사용 순서]
#   ① 이 셀 실행 → 셀 아래(또는 https://….gradio.live 주소)에 웹 화면이 나타남
#   ② 수리 내역 .csv 파일과 정답 파일 repair_labels.csv 를 고르고 → [학습하기] (몇 초)
#   ③ 수리 기록을 입력하거나 예시를 누르고 → [분류하기] (Enter 가능)
#   끝낼 때: 셀 왼쪽의 ■(중지) 버튼
#
# [주의] 공개 주소(….gradio.live)는 주소를 아는 누구나 접속할 수 있음
# ==========================================================================


# ----- 0. 도구 설치 -----
# PIL(Pillow): Colab에 이미 설치된 이미지 도구, 지금 버전 번호를 알아내려고 불러옴
import PIL

# pip 설치 (-q: 메시지 줄이기)
# pillow=={PIL.__version__}: 설치 중에 Pillow 버전이 바뀌어 ImportError가 나는 것을 막으려고 지금 버전으로 고정
!pip install -q koreanize-matplotlib gradio pillow=={PIL.__version__}


# ----- 도구 불러오기 -----
import re                                         # 정규식(글자 패턴 찾기·바꾸기)
import pandas as pd                               # 표(DataFrame) 다루기
import matplotlib.pyplot as plt                   # 그래프 그리기
import koreanize_matplotlib                       # 불러오기만 하면 그래프 한글 글꼴(나눔고딕)이 설정됨
from sklearn.model_selection import train_test_split               # 학습용/평가용 나누기
from sklearn.feature_extraction.text import TfidfVectorizer        # 글자 조각 → TF-IDF 숫자
from sklearn.linear_model import LogisticRegression                # 분류 모델
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity             # 문서끼리 비슷한 정도(0~1)
import gradio as gr                               # 웹 화면 만들기

# 유형 순서 (표와 그래프에서 늘어놓을 순서)
ORDER = ["소모품 교환", "작업날 교환", "부품 수리·교체", "일반 정비·점검", "고장 증상"]


# ----- 1. 전처리 함수 -----
# ✏️ 표기 통일 사전 {바꿀 표현: 통일할 표현}
NORM = {
    "밸트": "벨트", "밧데리": "배터리", "바테리": "배터리", "밧테리": "배터리", "축전지": "배터리", "춛전지": "배터리",
    "베아링": "베어링", "베어린": "베어링", "로타리": "로터리", "로터리발": "로터리날",
    "휠타": "필터", "휠터": "필터", "필타": "필터", "에이크리너": "에어클리너", "에어크리너": "에어클리너", "에어크린너": "에어클리너",
    "스폰치": "스펀지", "스폰지": "스펀지", "스펀치": "스펀지", "캬브레다": "기화기", "카브레다": "기화기",
    "죠인트": "조인트", "쪼인트": "조인트", "구리스": "그리스", "니뿔": "니플", "모타": "모터", "교한": "교환", "교체": "교환",
}


# 기록 하나 → 정리된 문서
def clean(text):
    text = re.sub(r"\[.*?\]", " ", text)                             # [수리중], [폐기] 같은 대괄호 부분 지우기 (.*? : 가장 짧게)
    text = re.sub(r"\S+님", " ", text)                                # '○○○님' 처럼 '님'으로 끝나는 말(사람 이름) 지우기
    text = re.sub(r"(?<![A-Za-z])(LB|B)\s*-?\s*\d+", " 벨트 ", text)  # 벨트 규격(B-58, LB-60) → '벨트'
    for old, new in NORM.items():                                     # 사전의 (바꿀 표현, 통일할 표현)을 하나씩 꺼내서
        text = text.replace(old, new)                                 # 글자 그대로 바꾸기
    text = re.sub(r"\d+\s*(개|종|리터|EA|ea)", " ", text)               # 수량(12개, 3종, 2EA) 지우기
    text = re.sub(r"[A-Za-z]*-?\d+[A-Za-z]*", " ", text)              # 숫자가 든 규격·부품 번호(6307, sa-124) 지우기
    text = re.sub(r"[^가-힣A-Za-z]", " ", text)                        # 한글·영어가 아닌 글자(기호, 괄호, 마침표) → 공백
    text = re.sub(r"내용없음|완료", " ", text)                          # 뜻 없는 말 지우기
    return re.sub(r"\s+", " ", text).strip()                          # 여러 칸 공백을 한 칸으로, 앞뒤 공백 지우기


# CSV 파일 읽기: utf-8로 먼저 읽어 보고, 한글이 깨져 오류가 나면 cp949(공공기관·엑셀 파일에 흔한 방식)로 다시 읽기
def read_csv(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")


# ----- 2. 학습 함수: [학습하기] 버튼 -----
def train(raw_file, label_file):
    # 파일 두 개를 다 고르지 않았으면 안내 메시지만 돌려줌 (나머지 출력은 비움)
    if raw_file is None or label_file is None:
        return "수리 내역 .csv 파일과 정답 파일(repair_labels.csv)을 모두 선택하세요.", None, None, None

    # 수리 내역 읽기 → 원래 줄 번호(0, 1, 2, …)를 기록 번호 ID로 사용 (정답 파일의 ID와 같은 번호)
    raw = read_csv(raw_file)
    raw["ID"] = raw.index

    # 비고가 빈 칸(NaN)이면 빈 글자로 바꾼 뒤 수리내역과 이어 붙이기 → 정제 함수 적용
    raw["원문"] = raw["수리내역"].fillna("") + " " + raw["비고"].fillna("")
    raw["문서"] = raw["원문"].apply(clean)

    # 정답 파일(열: ID, 유형)과 합치기: 두 표에 모두 있는 ID만 남김 (정비 기록이 아닌 것은 정답 파일에 없어서 빠짐)
    merged = raw.merge(read_csv(label_file), on="ID", how="inner")

    # 정리된 문서가 똑같은 기록은 하나만 남기고, 글자가 하나도 안 남은 문서는 빼기
    # (같은 문장이 학습·평가에 동시에 들어가 점수가 부풀려지는 것을 막음)
    data = merged.drop_duplicates("문서")
    data = data[data["문서"] != ""].reset_index(drop=True)
    if len(data) == 0:
        return "기록 번호(ID)와 정답 파일이 맞지 않습니다. 파일을 확인하세요.", None, None, None

    # 80%는 학습용, 20%는 평가용 (stratify: 유형 비율을 똑같이, random_state: 매번 같게)
    X_train, X_test, y_train, y_test = train_test_split(
        data["문서"], data["유형"], test_size=0.2, random_state=42, stratify=data["유형"])

    # 글자 단위 TF-IDF: 띄어쓰기 안에서 글자 2~3개씩 겹치게 자른 조각 (붙여 쓰기·오타에 강함)
    # 학습용으로 조각 사전을 만들고(fit) 숫자로 바꾸기, 평가용은 바꾸기만
    tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
    A_train = tfidf.fit_transform(X_train)
    A_test = tfidf.transform(X_test)

    # 로지스틱 회귀 학습 (class_weight="balanced": 개수가 적은 유형을 틀리면 더 크게 벌점)
    model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(A_train, y_train)

    # 평가용으로 예측하고 정확도·macro-F1 계산 (macro-F1: 유형마다 F1을 구해 똑같은 비중으로 평균)
    pred = model.predict(A_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="macro")

    # 유형별 성적표 (output_dict=True: 사전으로 받아 표로 바꿈, .T: 행과 열 바꾸기)
    # → ORDER 유형만 남기고 소수 셋째 자리로 반올림, 열 이름을 한글로
    report = pd.DataFrame(classification_report(y_test, pred, labels=ORDER, output_dict=True)).T
    report = report.loc[ORDER].round(3).reset_index().rename(columns={"index": "유형", "support": "평가 문서 수"})
    report["평가 문서 수"] = report["평가 문서 수"].astype(int)            # 80.0 → 80 (소수점 없애기)

    # 혼동 행렬 그림 (가로 = 예측, 세로 = 정답, 대각선 = 맞힌 문서 수)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=ORDER, cmap="Blues",
                                            colorbar=False, ax=ax, xticks_rotation=30)
    ax.set_xlabel("예측")
    ax.set_ylabel("정답")
    ax.set_title("혼동 행렬 (평가용)")
    plt.tight_layout()                                     # 글자가 잘리지 않게 여백 자동 조정
    plt.close(fig)                                         # 셀 아래에 따로 그려지지 않게 닫기 (웹 화면에만 표시)

    # 처리 결과 메시지 ({:,}: 천 단위 쉼표, {:.1%}: 백분율 소수 첫째 자리)
    message = (f"기록 {len(merged):,}건 → 중복 제거 후 문서 {len(data):,}개로 학습 완료 "
               f"(학습 {len(X_train):,} / 평가 {len(X_test):,}). 평가용 정확도 {acc:.1%}, macro-F1 {f1:.3f}")

    # 검색용: 정답이 붙은 전체 기록(중복 포함)을 TF-IDF 숫자로 바꿔 둠
    A_all = tfidf.transform(merged["문서"])

    # 메시지, 성적표, 혼동 행렬, 보관함(응답 함수가 쓸 도구와 데이터 묶음)을 돌려줌
    return message, report, fig, (tfidf, model, merged, A_all)


# ----- 3. 응답 함수: [분류하기] 버튼 -----
def ask(record, store):
    # 학습 전이거나 기록이 비어 있으면 안내 메시지
    if store is None:
        return "먼저 파일 두 개를 고르고 [학습하기]를 누르세요.", None, None
    if not record.strip():
        return "수리 기록을 입력하세요.", None, None

    # 보관함에서 TF-IDF 도구, 모델, 전체 기록, 전체 기록 숫자 행렬 꺼내기
    tfidf, model, merged, A_all = store

    # 새 기록도 학습 때와 같은 정제 → TF-IDF 숫자로 바꾸기 (fit 없이 transform만)
    doc = clean(record)
    A_new = tfidf.transform([doc])

    # 유형별 확률 (predict_proba: 0~1 → ×100 해서 %), 모델의 유형 순서를 ORDER 순서로 다시 정렬
    proba = pd.Series(model.predict_proba(A_new)[0] * 100, index=model.classes_)[ORDER]
    best = proba.idxmax()                                  # 확률이 가장 큰 유형

    # 응답 글
    answer = (f"예측 유형: {best} ({proba[best]:.1f}%)\n"
              f"정리된 기록: {doc if doc else '(없음 — 남은 글자가 없어 예측이 불확실함)'}")

    # 유형별 확률 표
    proba_table = proba.round(1).reset_index()
    proba_table.columns = ["유형", "확률(%)"]

    # 검색 결과: 새 기록과 TF-IDF 코사인 유사도가 가장 높은 과거 기록 5개 (같은 문서는 하나만)
    merged = merged.assign(유사도=cosine_similarity(A_new, A_all)[0].round(3))     # 기록마다 비슷한 정도(0~1)
    similar = (merged[merged["유사도"] > 0]                                         # 겹치는 조각이 없는 기록은 빼기
               .sort_values("유사도", ascending=False)                              # 유사도 큰 순서로 정렬
               .drop_duplicates("문서")                                             # 같은 문서는 하나만
               .head(5)[["수리일자", "원문", "유형", "금액", "유사도"]])

    return answer, proba_table, similar


# ----- 4. 웹 화면 -----
# gr.Blocks(): 웹 화면 전체 틀, with 블록 안의 부품이 위에서부터 차례로 배치됨
with gr.Blocks() as demo:
    gr.Markdown("# 🚜 농기계 고장·정비 기록 문서 분류")
    gr.Markdown("짧은 수리 기록을 정비 유형(소모품 교환 / 작업날 교환 / 부품 수리·교체 / 일반 정비·점검 / 고장 증상)으로 "
                "자동 분류하고, 비슷한 과거 수리 기록을 찾아 줍니다. (표기 통일 + 글자 단위 TF-IDF + 로지스틱 회귀)")

    # 보이지 않는 보관함: train이 만든 (TF-IDF, 모델, 기록, 숫자 행렬)을 담아 ask에 넘겨줌
    store = gr.State()

    # ===== ① 파일 선택 =====
    gr.Markdown("## ① 파일 선택")
    with gr.Row():                                         # gr.Row(): 안의 부품을 가로로 나란히
        raw_file = gr.File(label="수리 내역 파일 (.csv)", file_types=[".csv"])
        label_file = gr.File(label="정답 파일 (repair_labels.csv)", file_types=[".csv"])
    train_btn = gr.Button("학습하기", variant="primary")
    status = gr.Textbox(label="처리 결과")

    # ===== ② 학습 결과 =====
    gr.Markdown("## ② 학습 결과 (평가용 20%)")
    report = gr.Dataframe(label="유형별 성적표 (precision 정밀도 · recall 재현율 · f1-score)")
    cm_plot = gr.Plot(label="혼동 행렬")

    # ===== ③ 기록 분류와 비슷한 기록 검색 =====
    gr.Markdown("## ③ 수리 기록 분류하기")
    record = gr.Textbox(label="수리 기록", placeholder="예) 로타리칼 24개 교환")
    # ✏️ 예시 기록: 누르면 입력 칸이 채워짐
    gr.Examples(examples=["엔진오일 및 휠타 교환", "로타리칼 24개 교환", "유압호스 파손으로 교환",
                          "시동불량", "정기점검"], inputs=record)
    ask_btn = gr.Button("분류하기", variant="primary")
    answer = gr.Textbox(label="응답", lines=2)
    with gr.Row():
        proba_table = gr.Dataframe(label="유형별 확률", scale=1)
        similar = gr.Dataframe(label="검색 결과: 비슷한 과거 수리 기록 5개", wrap=True, scale=3,
                               column_widths=["15%", "45%", "18%", "12%", "10%"])   # 열 너비 (수리일자, 원문, 유형, 금액, 유사도)

    # 버튼과 함수 연결: 함수가 돌려준 값이 outputs 부품에 순서대로 표시됨
    train_btn.click(train, inputs=[raw_file, label_file], outputs=[status, report, cm_plot, store])
    ask_btn.click(ask, inputs=[record, store], outputs=[answer, proba_table, similar])
    record.submit(ask, inputs=[record, store], outputs=[answer, proba_table, similar])   # Enter로도 실행

# 웹 화면 실행 (debug=True: 셀이 계속 실행되며 오류를 셀 아래에 보여 줌)
demo.launch(debug=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 13.0 MB/s eta 0:00:00
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a72385adf4a670e5bc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
